# ReplyWise x TRIBE v2: Neuroscience-Informed Email Scoring Research

## Understanding What Makes Emails Compelling

This notebook runs the complete TRIBE v2 email research pipeline **in Google Colab with zero local setup**.

### How It Works

We use [Meta's TRIBE v2](https://github.com/facebookresearch/tribev2) — a brain encoding model trained on fMRI data — to predict neural responses to email text. By comparing which patterns trigger stronger brain activation in **replied** vs **ignored** emails, we uncover the neuroscience of engagement.

**Key regions analyzed:**
- **Left language network**: Core language comprehension
- **Prefrontal cortex**: Decision-making and attention
- **Emotional regions**: Salience and motivation to respond
- **Memory systems**: Novelty detection and memorability

### Output
This research generates `scoring_weights.json` — neuroscience-informed recommendations for email scoring engines like ReplyWise.

### License
**CC-BY-NC-4.0** for non-commercial research use. The TRIBE v2 model itself is used under research terms. The derived **scoring weights** are original work and can be used commercially.

## Cell 2: Install Dependencies

In [ ]:
# Install TRIBE v2 and dependencies
import subprocess
import sys

print("📦 Installing dependencies...\n")

# Install tribev2 from GitHub
print("Installing TRIBE v2...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/facebookresearch/tribev2.git@main#egg=tribev2[plotting]"])

# Install other dependencies
print("Installing additional packages...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy", "matplotlib", "seaborn", "pandas"])

print("\n✅ Dependencies installed!")
print("\n⚠️  Next step: You'll need to login to HuggingFace to access the TRIBE v2 model.")
print("   The model uses LLaMA 3.2 as a text encoder, which requires authentication.")
print("\n   Run this command when prompted in the next cell:")
print("   huggingface-cli login")

## Cell 3: Prepare Your Email Dataset

### Folder Structure
The pipeline expects emails organized like this:
```
emails/
    replied/
        email1.txt
        email2.txt
    ignored/
        email3.txt
        email4.txt
```

### Two Options
**Option A**: Upload your own emails manually (next cell)

**Option B**: Use the Enron dataset (~600k emails from Enron employees, public domain) — cell after that

Each `.txt` file should contain the **plain text body** of one email (no headers needed).

## Cell 4: Option A - Upload Your Own Emails

In [ ]:
# Option A: Upload your own email files

from pathlib import Path
import os

# Create folder structure
emails_dir = Path("/content/emails")
replied_dir = emails_dir / "replied"
ignored_dir = emails_dir / "ignored"

replied_dir.mkdir(parents=True, exist_ok=True)
ignored_dir.mkdir(parents=True, exist_ok=True)

print("📁 Email folder structure created:")
print(f"   {emails_dir}/")
print(f"       replied/   ← Upload emails that got replies")
print(f"       ignored/   ← Upload emails that were ignored")
print()
print("To upload your emails:")
print("  1. Click the folder icon (📁) in the left sidebar")
print("  2. Navigate to /content/emails/replied or /content/emails/ignored")
print("  3. Right-click → Upload files")
print("  4. Select your .txt email files")
print()
print("Each file should be a plain text email body (one email per file).")

## Cell 5: Option B - Download and Parse Enron Dataset

In [ ]:
# Option B: Download and parse Enron dataset
# The Enron Email Corpus: ~600k emails from 150 Enron employees (public domain)
# Source: https://www.cs.cmu.edu/~enron/

import subprocess
import os
from pathlib import Path
import email
from email.parser import BytesParser
from email.policy import default as default_policy
import re
from collections import defaultdict
from typing import Dict, List, Optional, Tuple
import json
from datetime import datetime

print("📥 Downloading Enron dataset (~1.7GB)...")
print("   This may take 5-10 minutes. Grab some coffee!\n")

# Download Enron dataset
dataset_path = Path("/content/enron_data")
dataset_path.mkdir(exist_ok=True)

enron_tar = dataset_path / "enron_mail_20150507.tar.gz"
enron_dir = dataset_path / "enron_mail_20150507"

if not enron_dir.exists():
    os.chdir(str(dataset_path))
    subprocess.run(
        ["wget", "-q", "https://www.cs.cmu.edu/~enron/enron_mail_20150507.tar.gz"],
        check=True
    )
    print("📦 Extracting archive...\n")
    subprocess.run(
        ["tar", "-xzf", "enron_mail_20150507.tar.gz"],
        check=True
    )
    print("✅ Enron dataset downloaded and extracted\n")
else:
    print("✅ Enron dataset already exists\n")

# ═══════════════════════════════════════════════════════════════════════════════
# Email parsing and filtering logic (inlined from parse_enron.py)
# ═══════════════════════════════════════════════════════════════════════════════

# Auto-generated email patterns to filter out
AUTO_GENERATED_PATTERNS = [
    r'calendar\s+request',
    r'meeting\s+request',
    r'meeting\s+accepted',
    r'meeting\s+declined',
    r'meeting\s+cancelled',
    r'out\s+of\s+office',
    r'auto-reply',
    r'automated',
    r'calendar invitation',
    r'time zone information',
    r'delivery\s+status\s+notification',
    r'undeliverable',
    r'bounced',
    r'failure notice',
    r'mailer-daemon',
]

# Anonymization patterns
ANONYMIZATION_PATTERNS = {
    'email': (r'\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b', '[EMAIL]'),
    'phone': (r'\b(?:\+?1[-.]?)?\(?[0-9]{3}\)?[-.]?[0-9]{3}[-.]?[0-9]{4}\b', '[PHONE]'),
    'url': (r'https?://[^\s]+', '[URL]'),
    'ip': (r'\b(?:[0-9]{1,3}\.){3}[0-9]{1,3}\b', '[IP]'),
}

def is_auto_generated(subject: str, body: str) -> bool:
    """Check if email is auto-generated."""
    full_text = (subject + ' ' + body).lower()
    for pattern in AUTO_GENERATED_PATTERNS:
        if re.search(pattern, full_text, re.IGNORECASE):
            return True
    return False

def remove_quoted_text(body: str) -> str:
    """Remove quoted/forwarded email text."""
    lines = body.split('\n')
    result = []
    for line in lines:
        if line.strip().startswith('>') or line.strip().startswith('|'):
            continue
        result.append(line)
    return '\n'.join(result).strip()

def remove_signature(body: str) -> str:
    """Remove email signature."""
    if '\n--\n' in body:
        body = body.split('\n--\n')[0]
    elif '\n--' in body:
        body = body.split('\n--')[0]
    return body.strip()

def clean_email_body(body: str) -> str:
    """Clean email body: remove signatures, quoted text, extra whitespace."""
    body = re.sub(r'<[^>]+>', '', body)
    body = remove_signature(body)
    body = remove_quoted_text(body)
    body = re.sub(r'\n\s*\n\s*\n+', '\n\n', body)
    body = re.sub(r'[ \t]+', ' ', body)
    return body.strip()

def anonymize_text(text: str) -> str:
    """Replace PII with placeholders."""
    for pii_type, (pattern, replacement) in ANONYMIZATION_PATTERNS.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text

def extract_email_text(email_msg) -> Tuple[str, str]:
    """Extract subject and body from email message."""
    subject = email_msg.get('Subject', '(no subject)')
    body = ''
    
    if email_msg.is_multipart():
        for part in email_msg.walk():
            content_type = part.get_content_type()
            if content_type == 'text/plain':
                try:
                    body = part.get_content(errors='replace')
                    break
                except:
                    pass
            elif content_type == 'text/html' and not body:
                try:
                    body = part.get_content(errors='replace')
                except:
                    pass
    else:
        try:
            body = email_msg.get_content(errors='replace')
        except:
            body = str(email_msg.get_payload(decode=False))
    
    return subject.strip(), body.strip()

def parse_email_file(filepath: Path) -> Optional[Dict]:
    """Parse a single email file from Enron maildir."""
    try:
        with open(filepath, 'rb') as f:
            msg = BytesParser(policy=default_policy).parse(f)
        
        subject, body = extract_email_text(msg)
        return {
            'filepath': str(filepath),
            'subject': subject,
            'body': body,
            'sender': msg.get('From', 'unknown@enron.com'),
            'message_id': msg.get('Message-ID', ''),
            'in_reply_to': msg.get('In-Reply-To', ''),
            'references': msg.get('References', ''),
        }
    except Exception as e:
        return None

def filter_quality_emails(email_data: Dict) -> Tuple[bool, str]:
    """Filter email for quality (50-500 words, no spam, no auto-generated)."""
    subject = email_data['subject']
    body = email_data['body']
    cleaned_body = clean_email_body(body)
    
    word_count = len(cleaned_body.split())
    if word_count < 50:
        return False, f"too_short"
    if word_count > 500:
        return False, f"too_long"
    
    if is_auto_generated(subject, cleaned_body):
        return False, "auto_generated"
    
    if subject.lower().startswith('fwd:') and cleaned_body.count('\n') < 3:
        return False, "forwarded_only"
    
    lines = cleaned_body.strip().split('\n')
    if len(lines) < 3:
        return False, "just_signature"
    
    if not cleaned_body or len(cleaned_body.strip()) < 100:
        return False, "empty"
    
    return True, "quality"

class ThreadAnalyzer:
    """Analyze email threads to identify replied vs ignored emails."""
    
    def __init__(self):
        self.email_map = {}
    
    def is_replied(self, email_data: Dict, all_emails: List[Dict]) -> bool:
        """Check if an email got a reply."""
        msg_id = email_data['message_id']
        if not msg_id:
            return False
        
        sender = email_data['sender'].lower()
        
        for other in all_emails:
            if other['message_id'] == msg_id:
                continue
            
            in_reply = (other['in_reply_to'] or '').lower()
            references = (other['references'] or '').lower()
            
            if msg_id.lower() in in_reply or msg_id.lower() in references:
                other_sender = other['sender'].lower()
                if other_sender != sender:
                    return True
        
        return False

def discover_enron_users(enron_path: Path) -> List[Path]:
    """Find all user directories in Enron maildir."""
    users = []
    for item in enron_path.iterdir():
        if item.is_dir() and not item.name.startswith('.'):
            users.append(item)
    return sorted(users)[:10]  # Limit to 10 users for speed in Colab

def process_user_maildir(user_path: Path, max_emails: Optional[int] = None) -> List[Dict]:
    """Process all emails from a single user's maildir."""
    emails = []
    email_count = 0
    
    for root, dirs, files in os.walk(user_path):
        dirs[:] = [d for d in dirs if not d.startswith('.')]
        
        for filename in sorted(files):
            if max_emails and email_count >= max_emails:
                break
            
            filepath = Path(root) / filename
            if not filepath.is_file():
                continue
            
            email_data = parse_email_file(filepath)
            if email_data:
                emails.append(email_data)
                email_count += 1
    
    return emails

# ═══════════════════════════════════════════════════════════════════════════════
# Main Enron processing
# ═══════════════════════════════════════════════════════════════════════════════

print("🔗 Discovering Enron users...")
users = discover_enron_users(enron_dir)
print(f"   Found {len(users)} users\n")

print("📧 Parsing emails...")
all_emails = []
for idx, user_path in enumerate(users):
    print(f"   [{idx + 1}/{len(users)}] {user_path.name}...", end=' ', flush=True)
    user_emails = process_user_maildir(user_path, max_emails=500)
    all_emails.extend(user_emails)
    print(f"{len(user_emails)} emails")

print(f"\n✅ Total emails parsed: {len(all_emails)}\n")

# Analyze threads and categorize
print("🔗 Analyzing email threads...")
analyzer = ThreadAnalyzer()

replied_emails = []
ignored_emails = []
stats = defaultdict(int)

for email_data in all_emails:
    is_quality, reason = filter_quality_emails(email_data)
    if not is_quality:
        stats[f"filtered_{reason}"] += 1
        continue
    
    cleaned_body = clean_email_body(email_data['body'])
    anonymized_body = anonymize_text(cleaned_body)
    
    has_reply = analyzer.is_replied(email_data, all_emails)
    
    if has_reply:
        replied_emails.append(anonymized_body)
    else:
        ignored_emails.append(anonymized_body)

# Limit to 50 per category for Colab (to keep runtime reasonable)
max_per_category = 50
replied_emails = replied_emails[:max_per_category]
ignored_emails = ignored_emails[:max_per_category]

# Write to output directories
print(f"\n💾 Writing output files...")
emails_dir = Path("/content/emails")
emails_dir.mkdir(exist_ok=True)
replied_dir = emails_dir / "replied"
ignored_dir = emails_dir / "ignored"
replied_dir.mkdir(exist_ok=True)
ignored_dir.mkdir(exist_ok=True)

for idx, body in enumerate(replied_emails):
    (replied_dir / f'email_{idx:05d}.txt').write_text(body, encoding='utf-8')

for idx, body in enumerate(ignored_emails):
    (ignored_dir / f'email_{idx:05d}.txt').write_text(body, encoding='utf-8')

print(f"\n✅ Enron dataset processed!")
print(f"   Replied:  {len(replied_emails)} emails")
print(f"   Ignored:  {len(ignored_emails)} emails")
print(f"   Total:    {len(replied_emails) + len(ignored_emails)} emails ready for analysis")

## Cell 6: Step 2 - Run TRIBE v2 Analysis

This cell loads the TRIBE v2 model and analyzes all your emails to predict neural responses.

In [ ]:
# First-time HuggingFace login (required for TRIBE v2 model)
# You only need to do this once per session

print("🔐 HuggingFace Authentication")
print("="*60)
print()
print("TRIBE v2 uses LLaMA 3.2 as its text encoder, which requires")
print("authentication via HuggingFace.")
print()
print("Steps:")
print("  1. Go to https://huggingface.co/settings/tokens")
print("  2. Create a new token (READ permission is enough)")
print("  3. Copy your token and paste it when prompted below")
print()

# Uncomment the line below to authenticate
# !huggingface-cli login

print("\nTo continue without authentication, you can:")
print("  - Skip this and run the cells below (may prompt later)")
print("  - Or uncomment and run the huggingface-cli login above")

## Cell 7: Load Model and Analyze All Emails

In [ ]:
# Load TRIBE v2 model and analyze emails

import time
import json
from pathlib import Path
import numpy as np

# Brain region definitions (inlined from analyze_emails.py)
REGION_VERTEX_RANGES = {
    "left_language": (3000, 5000),      # Left hemisphere language network
    "right_language": (13000, 15000),   # Right hemisphere homologue
    "prefrontal": (0, 2000),            # Bilateral prefrontal
    "temporal": (5000, 7000),           # Temporal regions
    "parietal": (7000, 9000),           # Parietal (attention)
    "emotional": (9000, 11000),         # Limbic-adjacent cortex
    "memory": (11000, 13000),           # Medial temporal
    "visual_language": (15000, 17000),  # Visual word form area
    "motor_speech": (17000, 19000),     # Motor/premotor (speech production)
}

def extract_region_activations(preds):
    """Extract mean activation per brain region from vertex predictions."""
    mean_activation = np.mean(preds, axis=0)
    peak_activation = np.max(preds, axis=0)
    
    region_stats = {}
    for region_name, (start, end) in REGION_VERTEX_RANGES.items():
        end = min(end, len(mean_activation))
        if start >= len(mean_activation):
            continue
        region_data = mean_activation[start:end]
        peak_data = peak_activation[start:end]
        region_stats[region_name] = {
            "mean_activation": float(np.mean(region_data)),
            "peak_activation": float(np.max(peak_data)),
            "std_activation": float(np.std(region_data)),
            "activation_range": float(np.ptp(region_data)),
        }
    return region_stats

def compute_engagement_score(region_stats):
    """Compute overall neural engagement score from brain region activations."""
    weights = {
        "left_language": 0.20,       # Core language processing
        "right_language": 0.10,      # Prosody, context, humor
        "prefrontal": 0.15,          # Attention, decision-making
        "temporal": 0.15,            # Comprehension
        "parietal": 0.10,            # Attention allocation
        "emotional": 0.15,           # Emotional salience (drives action)
        "memory": 0.10,              # Memorability, novelty
        "visual_language": 0.03,     # Reading fluency
        "motor_speech": 0.02,        # Inner speech / rehearsal
    }
    
    score = 0
    for region, weight in weights.items():
        if region in region_stats:
            r = region_stats[region]
            region_score = (r["mean_activation"] * 0.6 + r["peak_activation"] * 0.4)
            score += region_score * weight
    
    return score

def analyze_email(model, text_path):
    """Full analysis of a single email text file."""
    try:
        df = model.get_events_dataframe(text_path=str(text_path))
        preds, segments = model.predict(events=df)
    except Exception as e:
        print(f"     ⚠️  Error analyzing {text_path.name}: {e}")
        return None
    
    region_stats = extract_region_activations(preds)
    engagement = compute_engagement_score(region_stats)
    
    text = text_path.read_text(encoding="utf-8", errors="replace")
    word_count = len(text.split())
    line_count = len(text.strip().splitlines())
    
    return {
        "file": text_path.name,
        "word_count": word_count,
        "line_count": line_count,
        "engagement_score": engagement,
        "region_activations": region_stats,
        "prediction_shape": list(preds.shape),
    }

# Load model
print("📦 Loading TRIBE v2 model (709MB)...")
print("   First run downloads from HuggingFace (~2 min)\n")

try:
    from tribev2 import TribeModel
    start = time.time()
    model = TribeModel.from_pretrained("facebook/tribev2", cache_folder="/content/tribe_cache")
    print(f"   ✅ Model loaded in {time.time() - start:.1f}s\n")
except Exception as e:
    print(f"   ❌ Error loading model: {e}")
    print("   Make sure you've authenticated with HuggingFace.")
    raise

# Analyze emails
emails_dir = Path("/content/emails")
replied_dir = emails_dir / "replied"
ignored_dir = emails_dir / "ignored"

print("🧠 Analyzing REPLIED emails...")
replied_results = []
for txt_file in sorted(replied_dir.glob("*.txt")):
    result = analyze_email(model, txt_file)
    if result:
        result["category"] = "replied"
        replied_results.append(result)
        print(f"   ✅ {txt_file.name:20} | Engagement: {result['engagement_score']:.4f}")

print(f"\n✅ Analyzed {len(replied_results)} replied emails\n")

print("🧠 Analyzing IGNORED emails...")
ignored_results = []
for txt_file in sorted(ignored_dir.glob("*.txt")):
    result = analyze_email(model, txt_file)
    if result:
        result["category"] = "ignored"
        ignored_results.append(result)
        print(f"   ✅ {txt_file.name:20} | Engagement: {result['engagement_score']:.4f}")

print(f"\n✅ Analyzed {len(ignored_results)} ignored emails")

if not replied_results or not ignored_results:
    print("\n⚠️  Warning: Need at least 1 email in each category to compare.")
else:
    print(f"\n✅ All emails analyzed! Ready to compare.")

## Cell 8: Step 3 - Results & Comparison

### Comparing Brain Activation Patterns

Now we compare the neural activation patterns between replied and ignored emails to identify which brain regions differentiate them.

In [ ]:
# Compare groups and generate insights

def compare_groups(replied_results, ignored_results):
    """Compare brain activation patterns between replied and ignored emails."""
    def avg_regions(results):
        all_regions = {}
        for r in results:
            for region, stats in r["region_activations"].items():
                if region not in all_regions:
                    all_regions[region] = []
                all_regions[region].append(stats["mean_activation"])
        return {k: np.mean(v) for k, v in all_regions.items()}
    
    replied_avg = avg_regions(replied_results)
    ignored_avg = avg_regions(ignored_results)
    
    comparison = {}
    for region in replied_avg:
        if region in ignored_avg:
            diff = replied_avg[region] - ignored_avg[region]
            pct_diff = (diff / abs(ignored_avg[region]) * 100) if ignored_avg[region] != 0 else 0
            comparison[region] = {
                "replied_mean": float(replied_avg[region]),
                "ignored_mean": float(ignored_avg[region]),
                "difference": float(diff),
                "percent_difference": float(pct_diff),
                "favors": "replied" if diff > 0 else "ignored",
            }
    
    return comparison

print("🧠 Comparing neural activation patterns...\n")
comparison = compare_groups(replied_results, ignored_results)

# Print summary
print("="*70)
print("📊 BRAIN ACTIVATION COMPARISON")
print("="*70)

# Overall engagement scores
replied_eng = [r["engagement_score"] for r in replied_results]
ignored_eng = [r["engagement_score"] for r in ignored_results]

avg_replied = np.mean(replied_eng)
avg_ignored = np.mean(ignored_eng)
diff = avg_replied - avg_ignored

print(f"\n📈 Overall Neural Engagement:")
print(f"   Replied emails  (n={len(replied_results)}): {avg_replied:.4f}")
print(f"   Ignored emails  (n={len(ignored_results)}): {avg_ignored:.4f}")
print(f"   Difference: {diff:+.4f} ({'replied higher ✅' if diff > 0 else 'ignored higher ⚠️'})")
print(f"   Effect: {abs(diff/avg_ignored*100):+.1f}%\n")

# Top differentiating regions
print(f"🎯 Top Brain Regions Differentiating Replied vs Ignored:\n")
sorted_regions = sorted(
    comparison.items(),
    key=lambda x: abs(x[1]["percent_difference"]),
    reverse=True
)

for region, data in sorted_regions[:5]:
    arrow = "↑" if data["favors"] == "replied" else "↓"
    favor_text = "REPLIED" if data["favors"] == "replied" else "IGNORED"
    print(f"  {arrow} {region:20} {data['percent_difference']:+7.1f}%  (favors {favor_text})")
    print(f"      Replied: {data['replied_mean']:8.5f}  |  Ignored: {data['ignored_mean']:8.5f}\n")

## Cell 9: Visualize Results

In [ ]:
# Visualize results

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Set style
sns.set_style("darkgrid")
plt.rcParams["figure.figsize"] = (16, 12)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("TRIBE v2 Email Analysis: Neuroscience-Informed Insights", fontsize=16, fontweight='bold')

# ─────────────────────────────────────────────────────────────────────────────
# Chart 1: Overall Engagement Score Comparison
# ─────────────────────────────────────────────────────────────────────────────

ax = axes[0, 0]

replied_eng = [r["engagement_score"] for r in replied_results]
ignored_eng = [r["engagement_score"] for r in ignored_results]

data_for_plot = pd.DataFrame({
    'engagement_score': replied_eng + ignored_eng,
    'category': ['Replied']*len(replied_eng) + ['Ignored']*len(ignored_eng)
})

sns.barplot(data=data_for_plot, x='category', y='engagement_score', ax=ax, palette=['#2ecc71', '#e74c3c'])
ax.set_title('Average Neural Engagement Score', fontweight='bold', fontsize=12)
ax.set_ylabel('Engagement Score')
ax.set_xlabel('')

# Add value labels on bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.4f')

# ─────────────────────────────────────────────────────────────────────────────
# Chart 2: Brain Region Activation Heatmap
# ─────────────────────────────────────────────────────────────────────────────

ax = axes[0, 1]

heatmap_data = pd.DataFrame({
    region: [comparison[region]['replied_mean'], comparison[region]['ignored_mean']]
    for region in comparison.keys()
}, index=['Replied', 'Ignored'])

sns.heatmap(heatmap_data, annot=True, fmt='.5f', cmap='RdYlGn', ax=ax, cbar_kws={'label': 'Mean Activation'})
ax.set_title('Brain Region Activation Patterns', fontweight='bold', fontsize=12)
ax.set_xlabel('')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

# ─────────────────────────────────────────────────────────────────────────────
# Chart 3: Engagement Score Distribution (Box Plot)
# ─────────────────────────────────────────────────────────────────────────────

ax = axes[1, 0]

sns.boxplot(data=data_for_plot, x='category', y='engagement_score', ax=ax, palette=['#2ecc71', '#e74c3c'])
sns.stripplot(data=data_for_plot, x='category', y='engagement_score', ax=ax, color='black', alpha=0.3, size=4)
ax.set_title('Engagement Score Distribution', fontweight='bold', fontsize=12)
ax.set_ylabel('Engagement Score')
ax.set_xlabel('')

# ─────────────────────────────────────────────────────────────────────────────
# Chart 4: Top Differentiating Regions
# ─────────────────────────────────────────────────────────────────────────────

ax = axes[1, 1]

sorted_regions = sorted(
    comparison.items(),
    key=lambda x: abs(x[1]["percent_difference"]),
    reverse=True
)[:6]

regions = [r[0] for r in sorted_regions]
diffs = [r[1]["percent_difference"] for r in sorted_regions]
colors = ['#2ecc71' if d > 0 else '#e74c3c' for d in diffs]

ax.barh(regions, diffs, color=colors)
ax.set_xlabel('% Difference (Replied - Ignored)', fontweight='bold')
ax.set_title('Top Differentiating Brain Regions', fontweight='bold', fontsize=12)
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)

# Add value labels
for i, (region, diff) in enumerate(zip(regions, diffs)):
    ax.text(diff, i, f' {diff:+.1f}%', va='center', ha='left' if diff > 0 else 'right', fontweight='bold')

plt.tight_layout()
plt.savefig('/content/tribe_analysis_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Visualization saved to /content/tribe_analysis_visualization.png")

## Cell 10: Generate Scoring Weight Recommendations

In [ ]:
# Generate scoring insights and weight recommendations

def generate_scoring_insights(comparison, replied_results, ignored_results):
    """Translate neural differences into actionable scoring insights."""
    insights = []
    
    # Engagement scores
    replied_scores = [r["engagement_score"] for r in replied_results]
    ignored_scores = [r["engagement_score"] for r in ignored_results]
    
    avg_replied = np.mean(replied_scores) if replied_scores else 0
    avg_ignored = np.mean(ignored_scores) if ignored_scores else 0
    
    insights.append({
        "metric": "overall_engagement",
        "replied_avg": float(avg_replied),
        "ignored_avg": float(avg_ignored),
        "difference_pct": float((avg_replied - avg_ignored) / abs(avg_ignored) * 100) if avg_ignored != 0 else 0,
        "interpretation": "Higher neural engagement correlates with replies"
    })
    
    # Per-region insights
    top_differentiators = sorted(
        comparison.items(),
        key=lambda x: abs(x[1]["percent_difference"]),
        reverse=True
    )
    
    for region, data in top_differentiators[:5]:
        insights.append({
            "metric": f"region_{region}",
            "replied_mean": data["replied_mean"],
            "ignored_mean": data["ignored_mean"],
            "percent_difference": data["percent_difference"],
            "favors": data["favors"],
            "interpretation": f"{'Higher' if data['favors'] == 'replied' else 'Lower'} {region} activation in replied emails"
        })
    
    # Word count analysis
    replied_wc = [r["word_count"] for r in replied_results]
    ignored_wc = [r["word_count"] for r in ignored_results]
    
    if replied_wc and ignored_wc:
        insights.append({
            "metric": "optimal_word_count",
            "replied_avg_words": float(np.mean(replied_wc)),
            "ignored_avg_words": float(np.mean(ignored_wc)),
            "interpretation": f"Replied emails average {np.mean(replied_wc):.0f} words vs {np.mean(ignored_wc):.0f} for ignored"
        })
    
    return insights

def generate_weight_recommendations(insights):
    """Convert neuroscience insights into scoring weight recommendations."""
    recommendations = {
        "meta": {
            "generated_at": datetime.now().isoformat(),
            "model": "TRIBE v2 (facebook/tribev2)",
            "license": "CC-BY-NC-4.0 (non-commercial research)",
            "note": "These weight recommendations are derived from neuroscience research. The weights themselves are original work and can be used commercially."
        },
        "scoring_adjustments": {}
    }
    
    for insight in insights:
        metric = insight["metric"]
        
        if "emotional" in metric:
            recommendations["scoring_adjustments"]["emotional_tone"] = {
                "current_weight": 0.10,
                "recommended_weight": 0.15 if insight.get("favors") == "replied" else 0.08,
                "reason": insight["interpretation"]
            }
        elif "prefrontal" in metric:
            recommendations["scoring_adjustments"]["clarity_structure"] = {
                "current_weight": 0.15,
                "recommended_weight": 0.18 if insight.get("favors") == "replied" else 0.12,
                "reason": insight["interpretation"]
            }
        elif "left_language" in metric:
            recommendations["scoring_adjustments"]["readability"] = {
                "current_weight": 0.12,
                "recommended_weight": 0.16 if insight.get("favors") == "replied" else 0.10,
                "reason": insight["interpretation"]
            }
        elif "memory" in metric:
            recommendations["scoring_adjustments"]["memorability_novelty"] = {
                "current_weight": 0.08,
                "recommended_weight": 0.12 if insight.get("favors") == "replied" else 0.06,
                "reason": insight["interpretation"]
            }
    
    return recommendations

print("\n" + "="*70)
print("🧬 GENERATING SCORING INSIGHTS")
print("="*70 + "\n")

insights = generate_scoring_insights(comparison, replied_results, ignored_results)
print(f"Generated {len(insights)} insights\n")

recommendations = generate_weight_recommendations(insights)

print("📊 SCORING WEIGHT RECOMMENDATIONS\n")
for adjustment, config in recommendations["scoring_adjustments"].items():
    print(f"{adjustment}:")
    print(f"  Current weight:     {config['current_weight']:.2f}")
    print(f"  Recommended weight: {config['recommended_weight']:.2f}")
    print(f"  Delta:              {config['recommended_weight'] - config['current_weight']:+.2f}")
    print(f"  Rationale:          {config['reason']}")
    print()

print("\n✅ Recommendations generated!")

## Cell 11: Export Results

In [ ]:
# Export results to JSON files

from pathlib import Path
import json
from datetime import datetime

output_dir = Path("/content/results")
output_dir.mkdir(exist_ok=True)

# Compile all results
all_results = {
    "meta": {
        "generated_at": datetime.now().isoformat(),
        "model": "TRIBE v2 (facebook/tribev2)",
        "replied_count": len(replied_results),
        "ignored_count": len(ignored_results),
    },
    "summary": {
        "overall_engagement_replied": float(np.mean([r["engagement_score"] for r in replied_results])),
        "overall_engagement_ignored": float(np.mean([r["engagement_score"] for r in ignored_results])),
        "engagement_difference_pct": float((np.mean([r["engagement_score"] for r in replied_results]) - np.mean([r["engagement_score"] for r in ignored_results])) / np.mean([r["engagement_score"] for r in ignored_results]) * 100) if ignored_results else 0,
    },
    "individual_results": replied_results + ignored_results,
    "group_comparison": comparison,
    "scoring_insights": insights,
    "weight_recommendations": recommendations,
}

# Save full analysis
results_path = output_dir / "tribe_analysis.json"
with open(results_path, "w") as f:
    json.dump(all_results, f, indent=2, default=str)
print(f"✅ Full analysis: {results_path}")

# Save weight recommendations
weights_path = output_dir / "scoring_weights.json"
with open(weights_path, "w") as f:
    json.dump(recommendations, f, indent=2)
print(f"✅ Weight recommendations: {weights_path}")

# Save a human-readable summary
summary_path = output_dir / "summary.txt"
with open(summary_path, "w") as f:
    f.write("TRIBE v2 EMAIL ANALYSIS SUMMARY\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Generated: {datetime.now().isoformat()}\n")
    f.write(f"Model: TRIBE v2 (facebook/tribev2)\n")
    f.write(f"License: CC-BY-NC-4.0 (non-commercial research)\n\n")
    
    f.write("DATASET\n")
    f.write("-" * 70 + "\n")
    f.write(f"Replied emails:  {len(replied_results)}\n")
    f.write(f"Ignored emails:  {len(ignored_results)}\n")
    f.write(f"Total analyzed:  {len(replied_results) + len(ignored_results)}\n\n")
    
    f.write("KEY FINDINGS\n")
    f.write("-" * 70 + "\n")
    f.write(f"Overall neural engagement (replied): {all_results['summary']['overall_engagement_replied']:.6f}\n")
    f.write(f"Overall neural engagement (ignored): {all_results['summary']['overall_engagement_ignored']:.6f}\n")
    f.write(f"Difference: {all_results['summary']['engagement_difference_pct']:+.1f}%\n\n")
    
    f.write("TOP BRAIN REGIONS\n")
    f.write("-" * 70 + "\n")
    sorted_regions = sorted(
        comparison.items(),
        key=lambda x: abs(x[1]["percent_difference"]),
        reverse=True
    )
    for region, data in sorted_regions[:5]:
        f.write(f"{region}: {data['percent_difference']:+.1f}% (favors {data['favors']})\n")
    
    f.write("\nWEIGHT RECOMMENDATIONS\n")
    f.write("-" * 70 + "\n")
    for adjustment, config in recommendations["scoring_adjustments"].items():
        f.write(f"{adjustment}: {config['current_weight']:.2f} -> {config['recommended_weight']:.2f}\n")

print(f"✅ Summary: {summary_path}\n")

print("📁 All results saved to /content/results/")
print("   - tribe_analysis.json (complete analysis)")
print("   - scoring_weights.json (for import into ReplyWise)")
print("   - summary.txt (human-readable overview)")
print("   - tribe_analysis_visualization.png (charts)")

# Display files for download
print("\n💾 Click the folder icon on the left to download your results!")

## Cell 12: Next Steps

### What You've Done

You've run the complete TRIBE v2 neuroscience-informed email analysis pipeline. Using Meta's brain encoding model trained on fMRI data, you've identified which email patterns trigger stronger neural engagement and how that correlates with reply behavior.

### Output Files

Three key files are ready for use:

1. **`scoring_weights.json`** — Neuroscience-informed weight recommendations for email scoring engines
2. **`tribe_analysis.json`** — Complete analysis with region-by-region activation data
3. **`summary.txt`** — Human-readable overview of findings

### Integration with ReplyWise

Feed `scoring_weights.json` into ReplyWise's scoring engine to apply neuroscience-informed adjustments:

```bash
# Example: Load and apply weights
replywise-score --weights scoring_weights.json --emails your_emails.csv
```

The weights provide evidence-based guidance on:
- **Emotional tone**: Neural response to emotional salience
- **Clarity/structure**: Prefrontal engagement and decision-making
- **Readability**: Language network activation
- **Memorability/novelty**: Memory system engagement

### Further Research

- Increase your dataset: Run with 100+ emails per category for stronger statistical signals
- Stratified analysis: Compare emails by subject domain, sender type, or recipient
- A/B testing: Validate recommendations by testing email variants with your audience
- Model updates: Retrain as you collect more replied/ignored email data

### Attribution

**TRIBE v2** is developed by Meta's Foundational AI Research (FAIR):
- GitHub: [facebookresearch/tribev2](https://github.com/facebookresearch/tribev2)
- Paper: "Scaling brain-inspired models of language with self-supervised learning" (pending publication)

### License

- **TRIBE v2 model**: CC-BY-NC-4.0 (non-commercial research use)
- **Derived scoring weights**: Original work, commercially usable

### Questions?

Join the ReplyWise waitlist at **[replywise.pro](https://replywise.pro)** to stay updated and connect with the research community.

---

**Thank you for using the TRIBE v2 research pipeline!** Your insights help advance the science of email engagement.